# Phase 2: Feature Engineering & SQL Database Loading


In [1]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

# Set display options
pd.set_option('display.max_columns', None)

### Step 1: Load Staged Data


In [2]:
df = pd.read_csv('cleaned_superstore.csv')

# Convert strings back to datetime
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
if 'ship_date' in df.columns:
    df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')

print(f"Loaded Staged Data. Shape: {df.shape}")

Loaded Staged Data. Shape: (9994, 21)


### Step 2: Feature Engineering



In [3]:
# 2.1 Date Extractions
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

# Calculate processing time
df['processing_days'] = (df['ship_date'] - df['order_date']).dt.days

# 2.2 Financial Engineering
if 'profit' in df.columns and 'sales' in df.columns:
    df['profit_margin'] = (df['profit'] / df['sales']).fillna(0).round(4)
    # Categorical profit indicator
    df['is_profitable'] = df['profit'] > 0

print("Feature Engineering Complete. Snapshot of new columns:")
df[['order_date', 'order_year', 'processing_days', 'profit_margin', 'is_profitable']].head()

Feature Engineering Complete. Snapshot of new columns:


,order_date,order_year,processing_days,profit_margin,is_profitable
0,2016-11-08,2016,3,0.1600,True
1,2016-11-08,2016,3,0.3000,True
2,2016-06-12,2016,4,0.4700,True
3,2015-10-11,2015,7,-0.4000,False
4,2015-10-11,2015,7,0.1125,True


### Step 3: SQL Database Connection & Data Loading


In [4]:
# DB credentials
db_user = 'postgres'
db_password = 'nosqldontcry'
db_host = 'localhost'
db_port = '5432'
db_name = 'bi_project'

# Create connection engine
connection_url = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_url)

try:
    # Test connection
    with engine.connect() as connection:
        print("Successfully connected to PostgreSQL database: bi_project!")
except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to PostgreSQL database: bi_project!


In [5]:
# Save to SQL Table
table_name = 'cleaned_sales_data'

try:
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"SUCCESS: Loaded {len(df)} rows into the '{table_name}' table in PostgreSQL!")
except Exception as e:
    print(f"Failed to load data: {e}")

SUCCESS: Loaded 9994 rows into the 'cleaned_sales_data' table in PostgreSQL!


In [6]:
# Save final CSV
df.to_csv('final_engineered_data.csv', index=False)
print("Final data WITH features successfully saved as CSV!")


Final data WITH features successfully saved as CSV!
